# Notebook 08 — Uncertainty Estimation

## Objective
Estimate predictive uncertainty using Monte Carlo Dropout and investigate whether prediction intervals can achieve useful empirical coverage.

## Monte Carlo Dropout
Dropout remains active during repeated inference passes. Each pass produces a prediction, and the variation across predictions is used as an uncertainty estimate.

## Main Workflow
1. Load the trained forecasting model.
2. Perform repeated forward passes with dropout active.
3. Calculate the prediction mean and variation.
4. Construct prediction intervals.
5. Measure empirical interval coverage.
6. Select a calibration factor using validation data.
7. Evaluate the calibrated intervals on held-out test data.

## Important Concepts

### Predictive Uncertainty
Represents variation or uncertainty in the model's predictions.

### Prediction Interval
An interval intended to contain the actual target at a specified rate.

### Coverage
The proportion of actual target values that fall inside the prediction intervals.

### Calibration
Adjusting the intervals so their empirical coverage better matches the intended coverage level.

## Previously Recorded Reconstructed Run
- Test MAE: 6.4634
- Test RMSE: 11.2274
- Test MAPE: 13.8689%
- Raw validation coverage: 38.29%
- Raw test coverage: 36.24%
- Calibration factor: 12.6274
- Calibrated test coverage: 94.79%
- Mean calibrated interval width: 40.77

These results differ from an earlier run that used a calibration factor of 22.5. Verify which run and checkpoint are authoritative before presenting exact figures.

## Interpretation
Calibration brought test coverage close to the intended 95% level in the reconstructed run, but it also widened the intervals.

Coverage and interval width should be considered together. Very wide intervals may achieve good coverage while providing less precise information.

## Review Reminder
Be prepared to explain:
- Why dropout is repeated at inference.
- How prediction variation estimates uncertainty.
- What 95% coverage means.
- Why calibration is fitted using validation data.
- Why calibration on one dataset does not guarantee performance elsewhere.

# UrbanFlow AI — Uncertainty Estimation (Notebook 08)

**Purpose:** Estimate predictive uncertainty using MC Dropout for the Multi-Scale STGCN on METR-LA, evaluate uncertainty against forecasting error, and calibrate prediction intervals using validation data.

**This version embeds the exact `MultiScaleSTGCN` definition from the uploaded `06_multi_scale.ipynb`.** That model's `forward` method requires both `x` and the normalized adjacency matrix `A`; this notebook passes both.

Important: this notebook is reconstructed, not a byte-for-byte recovery of the earlier notebook. Use a checkpoint trained with this same model definition and kernel configuration. The experiment outputs may not reproduce historical results if the checkpoint/architecture differs.


In [30]:
# Optional GPU check (works in Colab; harmless if no NVIDIA GPU is available)
!nvidia-smi


Tue Oct 13 19:16:41 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   67C    P0             29W /   70W |     707MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

## 1. Imports, Google Drive, and paths

Expected processed files:
`train.npz`, `val.npz`, `test.npz`, `scaler.npz`, and `adj_METR-LA.pkl` under `/content/drive/MyDrive/urbanAi/data/processed/METR-LA`.

The default checkpoint is the combined-kernel model saved by the uploaded `06_multi_scale.ipynb`. If you want to evaluate a different checkpoint, update `CHECKPOINT_CANDIDATES` and make sure it was trained with the embedded architecture below.


In [31]:
from pathlib import Path
import sys, json, pickle, random
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

try:
    from google.colab import drive
    drive.mount("/content/drive")
except Exception:
    print("Google Drive mount skipped (not running in Colab).")

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", DEVICE)

PROCESSED_DIR = Path("/content/drive/MyDrive/urbanAi/data/processed/METR-LA")
CHECKPOINT_CANDIDATES = [
    Path("/content/experiments/multiscale_ablation/kernels_3_5/best_model.pt"),
    Path("/content/drive/MyDrive/urbanAi/experiments/multiscale_ablation/kernels_3_5/best_model.pt"),
    Path("/content/experiments/proposed/best_model.pt"),
    Path("/content/experiments/proposed/best_model.pth"),
]
INPUT_STEPS = 12
OUTPUT_STEPS = 3
NUM_NODES = 207
INPUT_CHANNELS = 1
HIDDEN_CHANNELS = 32
TEMPORAL_KERNELS = (3, 5)
DROPOUT = 0.2
BATCH_SIZE = 64
MC_SAMPLES = 30
NOMINAL_COVERAGE = 0.95

if not PROCESSED_DIR.exists():
    raise FileNotFoundError(
        f"Processed directory not found: {PROCESSED_DIR}. "
        "Mount Google Drive and verify the folder path."
    )
checkpoint_path = next((p for p in CHECKPOINT_CANDIDATES if p.exists()), None)
if checkpoint_path is None:
    raise FileNotFoundError(
        "No checkpoint found. Checked:\n" + "\n".join(map(str, CHECKPOINT_CANDIDATES))
    )
print("Checkpoint:", checkpoint_path)


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Device: cuda
Checkpoint: /content/experiments/multiscale_ablation/kernels_3_5/best_model.pt


## 2. Load processed splits and scaler

The loader supports common NPZ key names (`x`/`y`, `X`/`Y`, and `data` where applicable) and prints the available keys if the expected format is not found. It does not reshape tensors to force a match; the expected input/output shapes are checked explicitly.


In [32]:
def inspect_npz(path):
    z = np.load(path, allow_pickle=True)
    print(path.name, "keys:", z.files)
    return z

for split_name in ("train", "val", "test"):
    p = PROCESSED_DIR / f"{split_name}.npz"
    if not p.exists():
        raise FileNotFoundError(f"Missing split: {p}")
    inspect_npz(p)

scaler_path = PROCESSED_DIR / "scaler.npz"
if not scaler_path.exists():
    raise FileNotFoundError(f"Missing scaler: {scaler_path}")
scaler_npz = np.load(scaler_path, allow_pickle=True)
print("Scaler keys:", scaler_npz.files)

def get_array(npz, candidates):
    for key in candidates:
        if key in npz.files:
            return np.asarray(npz[key])
    raise KeyError(f"None of {candidates} found. Available keys: {npz.files}")

def load_split(name):
    z = np.load(PROCESSED_DIR / f"{name}.npz", allow_pickle=True)
    # Typical windowed split keys
    x = get_array(z, ["x", "X", "inputs", "data_x"])
    y = get_array(z, ["y", "Y", "targets", "data_y"])
    return x.astype(np.float32), y.astype(np.float32)

X_train, Y_train = load_split("train")
X_val, Y_val = load_split("val")
X_test, Y_test = load_split("test")

print("Train:", X_train.shape, Y_train.shape)
print("Val:  ", X_val.shape, Y_val.shape)
print("Test: ", X_test.shape, Y_test.shape)

def check_shapes(X, Y, split):
    expected_x_tail = (INPUT_STEPS, NUM_NODES, INPUT_CHANNELS)
    expected_y_tail = (OUTPUT_STEPS, NUM_NODES, INPUT_CHANNELS)
    if X.ndim != 4 or X.shape[1:] != expected_x_tail:
        raise ValueError(f"{split} X expected (N,{expected_x_tail}), got {X.shape}")
    if Y.ndim != 4 or Y.shape[1:] != expected_y_tail:
        raise ValueError(f"{split} Y expected (N,{expected_y_tail}), got {Y.shape}")
    if len(X) != len(Y):
        raise ValueError(f"{split}: X and Y sample counts differ")

for name, X, Y in [
    ("train", X_train, Y_train), ("val", X_val, Y_val), ("test", X_test, Y_test)
]:
    check_shapes(X, Y, name)


train.npz keys: ['X', 'Y']
val.npz keys: ['X', 'Y']
test.npz keys: ['X', 'Y']
Scaler keys: ['mean', 'std']
Train: (23976, 12, 207, 1) (23976, 3, 207, 1)
Val:   (3413, 12, 207, 1) (3413, 3, 207, 1)
Test:  (6841, 12, 207, 1) (6841, 3, 207, 1)


## 3. Define the exact model from `06_multi_scale.ipynb`, load adjacency and checkpoint

This architecture is copied from the model-definition cell in your uploaded notebook. It expects the normalized adjacency matrix as the second argument to `model(x, A)`.


In [33]:
class MultiScaleTemporalConv(nn.Module):
    def __init__(self, in_channels, out_channels, kernels=(3, 5), dropout=0.2):
        super().__init__()
        self.branches = nn.ModuleList([
            nn.Conv2d(
                in_channels, out_channels,
                kernel_size=(1, k), padding=(0, k // 2)
            )
            for k in kernels
        ])
        self.fusion = nn.Conv2d(out_channels * len(kernels), out_channels, kernel_size=1)
        self.activation = nn.ReLU()
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        x = x.permute(0, 3, 2, 1)  # B,C,N,T
        x = torch.cat([branch(x) for branch in self.branches], dim=1)
        x = self.dropout(self.activation(self.fusion(x)))
        return x.permute(0, 3, 2, 1)  # B,T,N,C


class GraphConvolution(nn.Module):
    def __init__(self, in_channels, out_channels):
        super().__init__()
        self.linear = nn.Linear(in_channels, out_channels)

    def forward(self, x, adjacency):
        return self.linear(torch.einsum("nm,btmc->btnc", adjacency, x))


class MultiScaleSTGCNBlock(nn.Module):
    def __init__(self, in_channels, hidden_channels, kernels=(3, 5), dropout=0.2):
        super().__init__()
        self.temporal1 = MultiScaleTemporalConv(in_channels, hidden_channels, kernels, dropout)
        self.graph = GraphConvolution(hidden_channels, hidden_channels)
        self.temporal2 = MultiScaleTemporalConv(hidden_channels, hidden_channels, kernels, dropout)
        self.activation = nn.ReLU()

    def forward(self, x, adjacency):
        x = self.temporal1(x)
        x = self.activation(self.graph(x, adjacency))
        return self.temporal2(x)


class MultiScaleSTGCN(nn.Module):
    def __init__(self, input_steps=12, output_steps=3, input_channels=1,
                 hidden_channels=32, kernels=(3, 5), dropout=0.2):
        super().__init__()
        self.block1 = MultiScaleSTGCNBlock(input_channels, hidden_channels, kernels, dropout)
        self.block2 = MultiScaleSTGCNBlock(hidden_channels, hidden_channels, kernels, dropout)
        self.temporal_head = nn.Linear(input_steps, output_steps)
        self.output_head = nn.Linear(hidden_channels, 1)

    def forward(self, x, adjacency):
        x = self.block1(x, adjacency)
        x = self.block2(x, adjacency)
        x = x.permute(0, 2, 3, 1)   # B,N,C,T
        x = self.temporal_head(x)   # B,N,C,H
        x = x.permute(0, 3, 1, 2)  # B,H,N,C
        return self.output_head(x)


# Load and row-normalize adjacency exactly as in the training notebook.
adj_path = PROCESSED_DIR / "adj_METR-LA.pkl"
if not adj_path.exists():
    raise FileNotFoundError(f"Adjacency file not found: {adj_path}")
with open(adj_path, "rb") as f:
    adj_data = pickle.load(f, encoding="latin1")
adj_np = np.asarray(
    adj_data[-1] if isinstance(adj_data, (tuple, list)) else adj_data,
    dtype=np.float32
)
if adj_np.shape != (NUM_NODES, NUM_NODES):
    raise ValueError(f"Expected adjacency {(NUM_NODES, NUM_NODES)}, got {adj_np.shape}")
degree = adj_np.sum(axis=1)
if np.any(degree <= 0):
    raise ValueError("Adjacency has a zero-degree row.")
adj_np = adj_np / degree[:, None]
A = torch.tensor(adj_np, dtype=torch.float32, device=DEVICE)
print("Adjacency loaded:", tuple(A.shape))

model = MultiScaleSTGCN(
    input_steps=INPUT_STEPS,
    output_steps=OUTPUT_STEPS,
    input_channels=INPUT_CHANNELS,
    hidden_channels=HIDDEN_CHANNELS,
    kernels=TEMPORAL_KERNELS,
    dropout=DROPOUT,
).to(DEVICE)

checkpoint = torch.load(checkpoint_path, map_location=DEVICE)
if isinstance(checkpoint, dict) and "model_state_dict" in checkpoint:
    state_dict = checkpoint["model_state_dict"]
elif isinstance(checkpoint, dict) and "state_dict" in checkpoint:
    state_dict = checkpoint["state_dict"]
else:
    state_dict = checkpoint

# Strict loading intentionally catches checkpoints from a different architecture.
model.load_state_dict(state_dict, strict=True)
print("Checkpoint loaded successfully.")


Adjacency loaded: (207, 207)
Checkpoint loaded successfully.


## 4. MC Dropout predictions

At inference time, keep the model in evaluation mode but activate only Dropout modules. This keeps normalization layers (if present) in evaluation mode while allowing stochastic forward passes. The output is expected to have shape `(samples, 3, 207, 1)`.


In [34]:
def enable_dropout_only(m):
    # Keep the whole model in eval mode; activate Dropout modules only.
    for layer in m.modules():
        if isinstance(layer, (nn.Dropout, nn.Dropout1d, nn.Dropout2d, nn.Dropout3d)):
            layer.train()

@torch.no_grad()
def mc_predict(model, X, adjacency, mc_samples=30, batch_size=64):
    all_draws = []
    model.eval()
    enable_dropout_only(model)
    for start in range(0, len(X), batch_size):
        xb = torch.as_tensor(X[start:start + batch_size], dtype=torch.float32, device=DEVICE)
        draws = []
        for _ in range(mc_samples):
            pred = model(xb, adjacency)
            draws.append(pred.detach().cpu().numpy())
        all_draws.append(np.stack(draws, axis=0))
    # Shape: (MC, N, horizon, nodes, channel)
    return np.concatenate(all_draws, axis=1)

val_draws = mc_predict(model, X_val, A, MC_SAMPLES, BATCH_SIZE)
test_draws = mc_predict(model, X_test, A, MC_SAMPLES, BATCH_SIZE)
print("Validation MC draws:", val_draws.shape)
print("Test MC draws:", test_draws.shape)

expected_tail = (OUTPUT_STEPS, NUM_NODES, INPUT_CHANNELS)
if val_draws.ndim != 5 or val_draws.shape[2:] != expected_tail:
    raise ValueError(f"Unexpected validation draw shape: {val_draws.shape}")
if test_draws.ndim != 5 or test_draws.shape[2:] != expected_tail:
    raise ValueError(f"Unexpected test draw shape: {test_draws.shape}")

val_mean = val_draws.mean(axis=0)
val_std = val_draws.std(axis=0, ddof=1)
test_mean = test_draws.mean(axis=0)
test_std = test_draws.std(axis=0, ddof=1)


Validation MC draws: (30, 3413, 3, 207, 1)
Test MC draws: (30, 6841, 3, 207, 1)


## 5. Convert predictions to original speed scale

This section reads the scaler parameters from `scaler.npz`. It supports common `mean`/`std` and `mean`/`scale` naming. If the project uses a custom scaler object or different parameter names, update this cell to match the preprocessing notebook. Never fit a new scaler on the test set.


In [35]:
# The scaler in the training notebook stores one mean/std value per sensor.
scaler_path = PROCESSED_DIR / "scaler.npz"
if not scaler_path.exists():
    raise FileNotFoundError(f"Scaler not found: {scaler_path}")
scaler_npz = np.load(scaler_path)
print("Scaler keys:", scaler_npz.files)
if "mean" not in scaler_npz.files or "std" not in scaler_npz.files:
    raise KeyError(f"Expected 'mean' and 'std' in scaler.npz; found {scaler_npz.files}")

scaler_mean = scaler_npz["mean"].astype(np.float32).reshape(1, 1, NUM_NODES, 1)
scaler_std = scaler_npz["std"].astype(np.float32).reshape(1, 1, NUM_NODES, 1)
if np.any(scaler_std <= 0):
    raise ValueError("Scaler standard deviation must be positive.")

def inverse_scale(a):
    return a * scaler_std + scaler_mean

Y_val_original = inverse_scale(Y_val)
Y_test_original = inverse_scale(Y_test)
val_mean_original = inverse_scale(val_mean)
val_std_original = val_std * scaler_std
test_mean_original = inverse_scale(test_mean)
test_std_original = test_std * scaler_std
print("Original-scale test predictions:", test_mean_original.shape)


Scaler keys: ['mean', 'std']
Original-scale test predictions: (6841, 3, 207, 1)


## 6. Forecast metrics and uncertainty/error relationship

MAPE can become unstable for very low target speeds. The threshold below excludes near-zero targets only from MAPE; MAE and RMSE use all valid values.


In [36]:
def metrics(y_true, y_pred, mape_epsilon=1.0):
    err = y_pred - y_true
    mae = np.mean(np.abs(err))
    rmse = np.sqrt(np.mean(err ** 2))
    valid = np.abs(y_true) >= mape_epsilon
    mape = np.mean(np.abs(err[valid] / y_true[valid])) * 100 if valid.any() else np.nan
    return {"MAE": float(mae), "RMSE": float(rmse), "MAPE_percent": float(mape)}

print("Test metrics:", metrics(Y_test_original, test_mean_original))

def corr_pair(a, b):
    a = np.asarray(a).ravel()
    b = np.asarray(b).ravel()
    mask = np.isfinite(a) & np.isfinite(b)
    a, b = a[mask], b[mask]
    if len(a) < 2 or np.std(a) == 0 or np.std(b) == 0:
        return np.nan, np.nan
    pearson = float(np.corrcoef(a, b)[0, 1])
    try:
        from scipy.stats import spearmanr
        spearman = float(spearmanr(a, b).statistic)
    except Exception:
        spearman = np.nan
    return pearson, spearman

test_abs_error = np.abs(test_mean_original - Y_test_original)
uncertainty = test_std_original
pearson, spearman = corr_pair(uncertainty, test_abs_error)
print("Uncertainty vs absolute error:")
print("  Pearson:", pearson)
print("  Spearman:", spearman)

print("\nPer-horizon metrics and uncertainty:")
for h in range(OUTPUT_STEPS):
    m = metrics(Y_test_original[:, h], test_mean_original[:, h])
    p, s = corr_pair(test_std_original[:, h], test_abs_error[:, h])
    print(
        f"{(h + 1) * 5:2d}-minute horizon | "
        f"MAE={m['MAE']:.4f}, RMSE={m['RMSE']:.4f}, "
        f"MAPE={m['MAPE_percent']:.4f}% | "
        f"mean uncertainty={test_std_original[:, h].mean():.6f}, "
        f"Pearson={p:.4f}, Spearman={s:.4f}"
    )

print("\nUncertainty distribution (original scale):")
for label, value in [
    ("mean", np.mean(uncertainty)),
    ("median", np.median(uncertainty)),
    ("std", np.std(uncertainty)),
    ("min", np.min(uncertainty)),
    ("p90", np.percentile(uncertainty, 90)),
    ("p95", np.percentile(uncertainty, 95)),
    ("p99", np.percentile(uncertainty, 99)),
    ("max", np.max(uncertainty)),
]:
    print(f"{label:>6}: {value:.6f}")


Test metrics: {'MAE': 6.463423252105713, 'RMSE': 11.227373123168945, 'MAPE_percent': 13.868892669677734}
Uncertainty vs absolute error:
  Pearson: 0.3848366388795667
  Spearman: 0.4636543085000798

Per-horizon metrics and uncertainty:
 5-minute horizon | MAE=6.1177, RMSE=10.4593, MAPE=13.1315% | mean uncertainty=1.676917, Pearson=0.4207, Spearman=0.4793
10-minute horizon | MAE=6.4450, RMSE=11.2504, MAPE=13.8437% | mean uncertainty=1.609955, Pearson=0.3839, Spearman=0.4633
15-minute horizon | MAE=6.8276, RMSE=11.9245, MAPE=14.6315% | mean uncertainty=1.556737, Pearson=0.3618, Spearman=0.4528

Uncertainty distribution (original scale):
  mean: 1.614537
median: 1.033005
   std: 1.287386
   min: 0.146738
   p90: 3.554080
   p95: 4.234672
   p99: 5.829903
   max: 13.869390


## 7. Raw nominal 95% prediction intervals

The raw interval is `mean ± 1.96 × MC standard deviation`. MC Dropout intervals are not guaranteed to be calibrated, so report their empirical coverage rather than assuming they achieve 95%.


In [37]:
Z_95 = 1.96

def interval_coverage(y, mean, std, factor=Z_95):
    lower = mean - factor * std
    upper = mean + factor * std
    covered = (y >= lower) & (y <= upper)
    return {
        "coverage_percent": float(covered.mean() * 100),
        "mean_width": float((upper - lower).mean()),
        "lower": lower,
        "upper": upper,
    }

raw_val = interval_coverage(Y_val_original, val_mean_original, val_std_original)
raw_test = interval_coverage(Y_test_original, test_mean_original, test_std_original)

print(f"Raw validation coverage: {raw_val['coverage_percent']:.4f}%")
print(f"Raw test coverage:       {raw_test['coverage_percent']:.4f}%")
print(f"Raw test mean width:     {raw_test['mean_width']:.6f}")


Raw validation coverage: 38.2939%
Raw test coverage:       36.2353%
Raw test mean width:     6.328987


## 8. Calibrate on validation only, then evaluate test once

Choose a multiplier `k` on the validation split to approach nominal 95% coverage. Freeze that multiplier and apply it to the held-out test split. Do not select `k` using test labels.


In [38]:
def coverage_for_factor(y, mean, std, factor):
    return float(((y >= mean - factor * std) & (y <= mean + factor * std)).mean())

target = NOMINAL_COVERAGE
# Broad deterministic search; denser around the selected region if needed.
candidate_factors = np.linspace(0.1, 50.0, 5000)
val_coverages = np.array([
    coverage_for_factor(Y_val_original, val_mean_original, val_std_original, k)
    for k in candidate_factors
])
best_idx = int(np.argmin(np.abs(val_coverages - target)))
calibration_factor = float(candidate_factors[best_idx])
val_calibrated_coverage = float(val_coverages[best_idx])

test_calibrated = interval_coverage(
    Y_test_original, test_mean_original, test_std_original,
    factor=calibration_factor
)

print(f"Calibration factor selected on validation: {calibration_factor:.4f}")
print(f"Validation coverage: {val_calibrated_coverage * 100:.4f}%")
print(f"Held-out test coverage: {test_calibrated['coverage_percent']:.4f}%")
print(f"Held-out test mean interval width: {test_calibrated['mean_width']:.6f}")

print("\nPer-horizon calibrated test coverage:")
for h in range(OUTPUT_STEPS):
    cov = coverage_for_factor(
        Y_test_original[:, h],
        test_mean_original[:, h],
        test_std_original[:, h],
        calibration_factor
    )
    width = 2 * calibration_factor * test_std_original[:, h]
    print(
        f"{(h + 1) * 5:2d}-minute horizon | "
        f"coverage={cov * 100:.4f}% | mean width={width.mean():.6f}"
    )


Calibration factor selected on validation: 12.6274
Validation coverage: 94.9969%
Held-out test coverage: 94.7860%
Held-out test mean interval width: 40.774811

Per-horizon calibrated test coverage:
 5-minute horizon | coverage=95.8514% | mean width=42.350243
10-minute horizon | coverage=94.7700% | mean width=40.659126
15-minute horizon | coverage=93.7366% | mean width=39.315117


## 9. Save outputs and reproducibility metadata

The output directory is separate from the training checkpoint. This cell saves the MC predictions, standard deviations, calibration factor, and summary metrics for later analysis. It does not overwrite model checkpoints.


In [39]:
OUTPUT_DIR = Path("/content/drive/MyDrive/urbanAi/experiments/uncertainty")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

summary = {
    "model": "MultiScaleSTGCN",
    "dataset": "METR-LA",
    "input_steps": INPUT_STEPS,
    "output_steps": OUTPUT_STEPS,
    "num_nodes": NUM_NODES,
    "kernels": list(TEMPORAL_KERNELS),
    "mc_samples": MC_SAMPLES,
    "dropout": DROPOUT,
    "checkpoint": str(checkpoint_path),
    "raw_test_coverage_percent": raw_test["coverage_percent"],
    "calibration_factor_selected_on_validation": calibration_factor,
    "calibrated_validation_coverage_percent": val_calibrated_coverage * 100,
    "calibrated_test_coverage_percent": test_calibrated["coverage_percent"],
    "calibrated_test_mean_interval_width": test_calibrated["mean_width"],
    "test_metrics": metrics(Y_test_original, test_mean_original),
    "uncertainty_error_pearson": pearson,
    "uncertainty_error_spearman": spearman,
}

np.savez_compressed(
    OUTPUT_DIR / "mc_dropout_predictions.npz",
    test_mean=test_mean_original,
    test_std=test_std_original,
    test_targets=Y_test_original,
    calibrated_lower=test_mean_original - calibration_factor * test_std_original,
    calibrated_upper=test_mean_original + calibration_factor * test_std_original,
    calibration_factor=np.array(calibration_factor),
)
with open(OUTPUT_DIR / "uncertainty_summary.json", "w") as f:
    json.dump(summary, f, indent=2)
print("Saved outputs to:", OUTPUT_DIR)
print(json.dumps(summary, indent=2))


Saved outputs to: /content/drive/MyDrive/urbanAi/experiments/uncertainty
{
  "model": "MultiScaleSTGCN",
  "dataset": "METR-LA",
  "input_steps": 12,
  "output_steps": 3,
  "num_nodes": 207,
  "kernels": [
    3,
    5
  ],
  "mc_samples": 30,
  "dropout": 0.2,
  "checkpoint": "/content/experiments/multiscale_ablation/kernels_3_5/best_model.pt",
  "raw_test_coverage_percent": 36.235344297348966,
  "calibration_factor_selected_on_validation": 12.627405481096218,
  "calibrated_validation_coverage_percent": 94.99686950482501,
  "calibrated_test_coverage_percent": 94.78598419447393,
  "calibrated_test_mean_interval_width": 40.774810791015625,
  "test_metrics": {
    "MAE": 6.463423252105713,
    "RMSE": 11.227373123168945,
    "MAPE_percent": 13.868892669677734
  },
  "uncertainty_error_pearson": 0.3848366388795667,
  "uncertainty_error_spearman": 0.4636543085000798
}


## Interpretation notes

- MC Dropout uncertainty is a proxy for predictive uncertainty, not a guarantee that a forecast is correct.
- If raw intervals under-cover, report that calibration was needed.
- The calibration multiplier is learned from validation data and may be large; report it transparently.
- Coverage near 95% on one held-out test split does not prove universal calibration or robustness under distribution shift.
- Do not claim uncertainty necessarily increases with forecast horizon unless the results support that claim.
- The project’s previously recorded run reported raw 95% interval coverage of about **19.39%**, and validation-calibrated test coverage of about **94.95%**. These are historical project results; the numbers printed by this notebook may differ if the architecture, checkpoint, scaler, or preprocessing differs.
